# Data Preprocessing M5 Forecasting Accuracy

Notebook ini mengimplementasikan **Tahap 4: Data Preprocessing** sesuai Bab 3 skripsi dan workflow penelitian.

## Prinsip metodologis
- Preprocessing dilakukan **setelah data splitting** untuk mencegah data leakage.
- Data penjualan diubah dari **wide** ke **long** setelah split utama terbentuk.
- Join dilakukan menggunakan `calendar.csv` dan `sell_prices.csv` pada split yang sudah dipisah.
- Semua mapping label encoding dan mean encoding di-fit **hanya dari training set utama**.
- One-hot encoding untuk `event_type` dibangun dari kategori yang muncul pada training set.

## Ruang lingkup notebook ini
- Menggabungkan `sales_train_main.csv` dan `sales_test_public.csv` dengan kalender dan harga.
- Melakukan cleaning, validasi, handling missing values, dan encoding kategorikal.
- Menyimpan artefak preprocessing untuk split utama.
- Tahap **feature engineering** seperti lag dan rolling akan dilakukan pada notebook berikutnya.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import math

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATASET_DIR = PROJECT_ROOT / "dataset"
SPLIT_DIR = DATASET_DIR / "splits"
PREPROCESSED_DIR = DATASET_DIR / "preprocessed"
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MODE = "smoke"  # ganti menjadi "full" untuk seluruh dataset
MAX_SERIES = 5 if MODE == "smoke" else None
CHUNK_SIZE = 5 if MODE == "smoke" else 500
COMPRESSION = "gzip"
SMOOTHING_M = 5.0

CALENDAR_PATH = DATASET_DIR / "calendar.csv"
SELL_PRICES_PATH = DATASET_DIR / "sell_prices.csv"
TRAIN_SPLIT_PATH = SPLIT_DIR / "sales_train_main.csv"
TEST_SPLIT_PATH = SPLIT_DIR / "sales_test_public.csv"
CALENDAR_SPLIT_LABELS_PATH = SPLIT_DIR / "calendar_split_labels.csv"

TRAIN_OUTPUT_PATH = PREPROCESSED_DIR / f"train_preprocessed_{MODE}.csv.gz"
TEST_OUTPUT_PATH = PREPROCESSED_DIR / f"test_preprocessed_{MODE}.csv.gz"
MAPPINGS_OUTPUT_PATH = PREPROCESSED_DIR / f"preprocessing_mappings_{MODE}.json"
SUMMARY_OUTPUT_PATH = PREPROCESSED_DIR / f"preprocessing_summary_{MODE}.json"

print(f"PROJECT_ROOT:        {PROJECT_ROOT}")
print(f"MODE:                {MODE}")
print(f"MAX_SERIES:          {MAX_SERIES}")
print(f"CHUNK_SIZE:          {CHUNK_SIZE}")
print(f"TRAIN_OUTPUT_PATH:   {TRAIN_OUTPUT_PATH}")
print(f"TEST_OUTPUT_PATH:    {TEST_OUTPUT_PATH}")


PROJECT_ROOT:        C:\Users\adeli\speedrun-lulus
MODE:                smoke
MAX_SERIES:          5
CHUNK_SIZE:          5
TRAIN_OUTPUT_PATH:   C:\Users\adeli\speedrun-lulus\dataset\preprocessed\train_preprocessed_smoke.csv.gz
TEST_OUTPUT_PATH:    C:\Users\adeli\speedrun-lulus\dataset\preprocessed\test_preprocessed_smoke.csv.gz


## 1. Load artefak hasil data splitting

Notebook ini menggunakan output tahap sebelumnya, bukan langsung melakukan preprocessing dari data mentah yang belum dipisah. Ini mengikuti urutan metodologis Bab 3.


In [3]:
calendar_raw = pd.read_csv(CALENDAR_PATH)
calendar_split_labels = pd.read_csv(CALENDAR_SPLIT_LABELS_PATH)
sell_prices = pd.read_csv(SELL_PRICES_PATH)
sales_train_wide = pd.read_csv(TRAIN_SPLIT_PATH)
sales_test_wide = pd.read_csv(TEST_SPLIT_PATH)

print("calendar_raw shape:       ", calendar_raw.shape)
print("calendar_split_labels:    ", calendar_split_labels.shape)
print("sell_prices shape:        ", sell_prices.shape)
print("sales_train_wide shape:   ", sales_train_wide.shape)
print("sales_test_wide shape:    ", sales_test_wide.shape)


calendar_raw shape:        (1969, 14)
calendar_split_labels:     (1969, 15)
sell_prices shape:         (6841121, 4)
sales_train_wide shape:    (30490, 1919)
sales_test_wide shape:     (30490, 34)


In [4]:
if MAX_SERIES is not None:
    sales_train_wide = sales_train_wide.head(MAX_SERIES).copy()
    sales_test_wide = sales_test_wide.head(MAX_SERIES).copy()

id_columns = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
train_day_columns = [col for col in sales_train_wide.columns if col.startswith("d_")]
test_day_columns = [col for col in sales_test_wide.columns if col.startswith("d_")]

assert train_day_columns[0] == "d_1"
assert train_day_columns[-1] == "d_1913"
assert test_day_columns[0] == "d_1914"
assert test_day_columns[-1] == "d_1941"

print("train series used:", len(sales_train_wide))
print("test series used: ", len(sales_test_wide))


train series used: 5
test series used:  5


## 2. Helper functions

Fungsi-fungsi berikut disusun agar preprocessing tetap modular dan bisa dipakai baik untuk mode `smoke` maupun `full`.


In [5]:
def add_group_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["store_cat"] = out["store_id"] + "|" + out["cat_id"]
    out["store_dept"] = out["store_id"] + "|" + out["dept_id"]
    return out


def validate_hierarchy_consistency(df: pd.DataFrame) -> None:
    hierarchy_checks = [
        ("item_id", "dept_id"),
        ("item_id", "cat_id"),
        ("store_id", "state_id"),
    ]
    for key_col, value_col in hierarchy_checks:
        max_unique = df.groupby(key_col)[value_col].nunique().max()
        assert max_unique == 1, f"Inkonsistensi hierarki: {key_col} -> {value_col}"


def build_label_mapping(values: pd.Series) -> dict[str, int]:
    unique_values = sorted(pd.Series(values).dropna().astype(str).unique().tolist())
    return {value: idx for idx, value in enumerate(unique_values)}


def build_mean_encoding_maps(
    sales_train_wide: pd.DataFrame,
    categorical_columns: list[str],
    train_day_columns: list[str],
    global_mean_sales: float,
    smoothing_m: float,
) -> dict[str, dict[str, float]]:
    base = add_group_columns(sales_train_wide[id_columns])
    sales_matrix = sales_train_wide[train_day_columns]
    base["sales_sum"] = sales_matrix.sum(axis=1).astype("float64")
    base["sales_count"] = float(len(train_day_columns))

    mean_maps: dict[str, dict[str, float]] = {}
    for col in categorical_columns:
        grouped = (
            base.groupby(col, dropna=False)[["sales_sum", "sales_count"]]
            .sum()
            .reset_index()
        )
        grouped["smoothed_mean"] = (
            grouped["sales_sum"] + smoothing_m * global_mean_sales
        ) / (grouped["sales_count"] + smoothing_m)
        mean_maps[col] = {
            str(key): float(value)
            for key, value in zip(grouped[col].astype(str), grouped["smoothed_mean"])
        }
    return mean_maps


def prepare_calendar(calendar_raw: pd.DataFrame, calendar_split_labels: pd.DataFrame) -> pd.DataFrame:
    calendar = calendar_raw.copy()
    calendar["date"] = pd.to_datetime(calendar["date"])
    calendar["event_name_1"] = calendar["event_name_1"].fillna("No_Event")
    calendar["event_name_2"] = calendar["event_name_2"].fillna("No_Event")
    calendar["event_type_1"] = calendar["event_type_1"].fillna("No_Type")
    calendar["event_type_2"] = calendar["event_type_2"].fillna("No_Type")
    calendar["dayofweek"] = calendar["date"].dt.dayofweek.astype("int8")
    calendar["is_weekend"] = calendar["dayofweek"].isin([5, 6]).astype("int8")
    calendar = calendar.merge(
        calendar_split_labels[["d", "main_split"]],
        on="d",
        how="left",
        validate="one_to_one",
    )
    return calendar


def build_encoding_artifacts(
    sales_train_wide: pd.DataFrame,
    calendar_prepared: pd.DataFrame,
    train_day_columns: list[str],
    smoothing_m: float,
) -> dict:
    train_ids = add_group_columns(sales_train_wide[id_columns])
    train_calendar = calendar_prepared[calendar_prepared["main_split"] == "train"].copy()

    global_mean_sales = float(sales_train_wide[train_day_columns].to_numpy().sum() / (len(sales_train_wide) * len(train_day_columns)))

    label_columns = [
        "item_id", "dept_id", "cat_id", "store_id", "state_id", "store_cat", "store_dept"
    ]
    label_mappings = {col: build_label_mapping(train_ids[col]) for col in label_columns}
    label_mappings["event_name_1"] = build_label_mapping(train_calendar["event_name_1"])
    label_mappings["event_name_2"] = build_label_mapping(train_calendar["event_name_2"])

    mean_encoding_columns = [
        "item_id", "dept_id", "cat_id", "store_id", "state_id", "store_cat", "store_dept"
    ]
    mean_encoding_maps = build_mean_encoding_maps(
        sales_train_wide=sales_train_wide,
        categorical_columns=mean_encoding_columns,
        train_day_columns=train_day_columns,
        global_mean_sales=global_mean_sales,
        smoothing_m=smoothing_m,
    )

    artifacts = {
        "mode": MODE,
        "max_series": MAX_SERIES,
        "global_mean_sales": global_mean_sales,
        "smoothing_m": smoothing_m,
        "label_mappings": label_mappings,
        "mean_encoding_maps": mean_encoding_maps,
        "event_type_1_categories": sorted(train_calendar["event_type_1"].astype(str).unique().tolist()),
        "event_type_2_categories": sorted(train_calendar["event_type_2"].astype(str).unique().tolist()),
    }
    return artifacts


def select_snap_value(df: pd.DataFrame) -> pd.Series:
    return pd.Series(
        np.select(
            [df["state_id"].eq("CA"), df["state_id"].eq("TX"), df["state_id"].eq("WI")],
            [df["snap_CA"], df["snap_TX"], df["snap_WI"]],
            default=0,
        ),
        index=df.index,
    ).astype("int8")


def write_dataframe(df: pd.DataFrame, output_path: Path, first_chunk: bool) -> None:
    mode = "w" if first_chunk else "a"
    header = first_chunk
    df.to_csv(output_path, mode=mode, header=header, index=False, compression=COMPRESSION)


def preprocess_split(
    wide_df: pd.DataFrame,
    split_name: str,
    day_columns: list[str],
    calendar_prepared: pd.DataFrame,
    sell_prices: pd.DataFrame,
    encoding_artifacts: dict,
    output_path: Path,
    chunk_size: int,
) -> dict:
    if output_path.exists():
        output_path.unlink()

    calendar_subset = calendar_prepared[calendar_prepared["d"].isin(day_columns)].copy()
    keep_calendar_cols = [
        "date", "wm_yr_wk", "weekday", "wday", "month", "year", "d", "event_name_1",
        "event_name_2", "event_type_1", "event_type_2", "snap_CA", "snap_TX", "snap_WI",
        "dayofweek", "is_weekend"
    ]
    calendar_subset = calendar_subset[keep_calendar_cols]

    label_columns = [
        "item_id", "dept_id", "cat_id", "store_id", "state_id", "store_cat", "store_dept", "event_name_1", "event_name_2"
    ]
    mean_encoding_columns = [
        "item_id", "dept_id", "cat_id", "store_id", "state_id", "store_cat", "store_dept"
    ]

    processed_series = len(wide_df)
    row_count = 0
    min_sales = math.inf
    max_sales = -math.inf
    first_chunk = True

    id_plus_group_cols = id_columns + ["store_cat", "store_dept"]

    for start in range(0, len(wide_df), chunk_size):
        chunk = wide_df.iloc[start:start + chunk_size].copy()
        chunk = add_group_columns(chunk)

        long_df = chunk.melt(
            id_vars=id_plus_group_cols,
            value_vars=day_columns,
            var_name="d",
            value_name="sales",
        )

        long_df = long_df.merge(calendar_subset, on="d", how="left", validate="many_to_one")
        long_df = long_df.merge(
            sell_prices,
            on=["store_id", "item_id", "wm_yr_wk"],
            how="left",
            validate="many_to_one",
        )

        long_df["date"] = pd.to_datetime(long_df["date"])
        long_df.sort_values(["store_id", "item_id", "date"], inplace=True)

        long_df["snap"] = select_snap_value(long_df)
        long_df["sell_price"] = (
            long_df.groupby(["store_id", "item_id"], sort=False)["sell_price"]
            .transform(lambda s: s.ffill().bfill())
        )
        long_df["sell_price"] = long_df["sell_price"].astype("float32")
        long_df["sales"] = pd.to_numeric(long_df["sales"], downcast="integer")
        long_df["split"] = split_name

        assert long_df[["id", "date"]].duplicated().sum() == 0, f"Duplikat ditemukan pada split {split_name}"
        assert (long_df["sales"] >= 0).all(), f"Sales negatif ditemukan pada split {split_name}"

        for col in label_columns:
            mapping = encoding_artifacts["label_mappings"][col]
            long_df[f"{col}_le"] = long_df[col].astype(str).map(mapping).fillna(-1).astype("int32")

        for col in mean_encoding_columns:
            mapping = encoding_artifacts["mean_encoding_maps"][col]
            long_df[f"{col}_mean_enc"] = (
                long_df[col].astype(str).map(mapping)
                .fillna(encoding_artifacts["global_mean_sales"])
                .astype("float32")
            )

        for event_col in ["event_type_1", "event_type_2"]:
            categories = encoding_artifacts[f"{event_col}_categories"]
            for category in categories:
                encoded_col = f"{event_col}__{category}"
                long_df[encoded_col] = long_df[event_col].eq(category).astype("int8")

        long_df["date"] = long_df["date"].dt.strftime("%Y-%m-%d")

        ordered_columns = [
            "id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "store_cat", "store_dept",
            "date", "d", "wm_yr_wk", "weekday", "wday", "dayofweek", "month", "year", "is_weekend",
            "event_name_1", "event_name_2", "event_type_1", "event_type_2",
            "event_name_1_le", "event_name_2_le", "snap", "sell_price", "sales", "split",
            "item_id_le", "dept_id_le", "cat_id_le", "store_id_le", "state_id_le", "store_cat_le", "store_dept_le",
            "item_id_mean_enc", "dept_id_mean_enc", "cat_id_mean_enc", "store_id_mean_enc", "state_id_mean_enc",
            "store_cat_mean_enc", "store_dept_mean_enc",
        ]

        event_type_one_hot_cols = []
        for event_col in ["event_type_1", "event_type_2"]:
            event_type_one_hot_cols.extend([f"{event_col}__{category}" for category in encoding_artifacts[f"{event_col}_categories"]])

        long_df = long_df[ordered_columns + event_type_one_hot_cols]

        write_dataframe(long_df, output_path, first_chunk=first_chunk)
        first_chunk = False

        row_count += len(long_df)
        min_sales = min(min_sales, int(long_df["sales"].min()))
        max_sales = max(max_sales, int(long_df["sales"].max()))

    return {
        "split": split_name,
        "processed_series": processed_series,
        "row_count": int(row_count),
        "min_sales": int(min_sales),
        "max_sales": int(max_sales),
        "output_path": str(output_path.resolve()),
    }


## 3. Validasi awal dan persiapan kalender

Langkah ini memastikan struktur hierarki data konsisten sebelum proses wide-to-long dan join dilakukan.


In [6]:
validate_hierarchy_consistency(sales_train_wide[id_columns])
validate_hierarchy_consistency(sales_test_wide[id_columns])

calendar_prepared = prepare_calendar(calendar_raw, calendar_split_labels)
calendar_prepared.head()


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,dayofweek,is_weekend,main_split
0,2011-01-29,11101,Saturday,1,1,2011,d_1,No_Event,No_Type,No_Event,No_Type,0,0,0,5,1,train
1,2011-01-30,11101,Sunday,2,1,2011,d_2,No_Event,No_Type,No_Event,No_Type,0,0,0,6,1,train
2,2011-01-31,11101,Monday,3,1,2011,d_3,No_Event,No_Type,No_Event,No_Type,0,0,0,0,0,train
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,No_Event,No_Type,No_Event,No_Type,1,1,0,1,0,train
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,No_Event,No_Type,No_Event,No_Type,1,0,1,2,0,train


## 4. Fit artefak encoding hanya dari training set

Sesuai Bab 3, mapping label encoding, kategori one-hot, dan mean encoding dibangun eksklusif dari training set utama.


In [7]:
encoding_artifacts = build_encoding_artifacts(
    sales_train_wide=sales_train_wide,
    calendar_prepared=calendar_prepared,
    train_day_columns=train_day_columns,
    smoothing_m=SMOOTHING_M,
)

print("global_mean_sales:", encoding_artifacts["global_mean_sales"])
print("event_type_1_categories:", encoding_artifacts["event_type_1_categories"])
print("event_type_2_categories:", encoding_artifacts["event_type_2_categories"])


global_mean_sales: 0.6814427600627287
event_type_1_categories: ['Cultural', 'National', 'No_Type', 'Religious', 'Sporting']
event_type_2_categories: ['Cultural', 'No_Type', 'Religious']


## 5. Jalankan preprocessing untuk training split

Tahap ini mencakup `melt`, join ke kalender dan harga, imputasi nilai hilang, lalu encoding kategori.


In [8]:
train_result = preprocess_split(
    wide_df=sales_train_wide,
    split_name="train",
    day_columns=train_day_columns,
    calendar_prepared=calendar_prepared,
    sell_prices=sell_prices,
    encoding_artifacts=encoding_artifacts,
    output_path=TRAIN_OUTPUT_PATH,
    chunk_size=CHUNK_SIZE,
)

train_result


{'split': 'train',
 'processed_series': 5,
 'row_count': 9565,
 'min_sales': 0,
 'max_sales': 15,
 'output_path': 'C:\\Users\\adeli\\speedrun-lulus\\dataset\\preprocessed\\train_preprocessed_smoke.csv.gz'}

## 6. Jalankan preprocessing untuk public test split

Transformasi yang diterapkan identik dengan training, tetapi seluruh mapping hanya menggunakan artefak yang telah di-fit dari training set.


In [9]:
test_result = preprocess_split(
    wide_df=sales_test_wide,
    split_name="test_public",
    day_columns=test_day_columns,
    calendar_prepared=calendar_prepared,
    sell_prices=sell_prices,
    encoding_artifacts=encoding_artifacts,
    output_path=TEST_OUTPUT_PATH,
    chunk_size=CHUNK_SIZE,
)

test_result


{'split': 'test_public',
 'processed_series': 5,
 'row_count': 140,
 'min_sales': 0,
 'max_sales': 6,
 'output_path': 'C:\\Users\\adeli\\speedrun-lulus\\dataset\\preprocessed\\test_preprocessed_smoke.csv.gz'}

## 7. Simpan mapping preprocessing dan ringkasan hasil


In [10]:
with open(MAPPINGS_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(encoding_artifacts, f, indent=2)

summary = {
    "source": "Outline Skripsi Bab 3 - Tahap Data Preprocessing",
    "mode": MODE,
    "max_series": MAX_SERIES,
    "compression": COMPRESSION,
    "notes": [
        "Join dilakukan setelah data splitting untuk mencegah leakage.",
        "Event kosong diisi dengan No_Event / No_Type.",
        "Sell price diisi dengan forward fill lalu backward fill per item-store pada indeks minggu kalender.",
        "Label encoding dan mean encoding di-fit hanya dari training set utama.",
        "Untuk cross-validation, mapping perlu di-fit ulang pada training fold masing-masing.",
    ],
    "train_result": train_result,
    "test_result": test_result,
    "artifacts": {
        "train_preprocessed": str(TRAIN_OUTPUT_PATH.resolve()),
        "test_preprocessed": str(TEST_OUTPUT_PATH.resolve()),
        "mappings": str(MAPPINGS_OUTPUT_PATH.resolve()),
        "summary": str(SUMMARY_OUTPUT_PATH.resolve()),
    },
}

with open(SUMMARY_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

summary


{'source': 'Outline Skripsi Bab 3 - Tahap Data Preprocessing',
 'mode': 'smoke',
 'max_series': 5,
 'compression': 'gzip',
 'notes': ['Join dilakukan setelah data splitting untuk mencegah leakage.',
  'Event kosong diisi dengan No_Event / No_Type.',
  'Sell price diisi dengan forward fill lalu backward fill per item-store pada indeks minggu kalender.',
  'Label encoding dan mean encoding di-fit hanya dari training set utama.',
  'Untuk cross-validation, mapping perlu di-fit ulang pada training fold masing-masing.'],
 'train_result': {'split': 'train',
  'processed_series': 5,
  'row_count': 9565,
  'min_sales': 0,
  'max_sales': 15,
  'output_path': 'C:\\Users\\adeli\\speedrun-lulus\\dataset\\preprocessed\\train_preprocessed_smoke.csv.gz'},
 'test_result': {'split': 'test_public',
  'processed_series': 5,
  'row_count': 140,
  'min_sales': 0,
  'max_sales': 6,
  'output_path': 'C:\\Users\\adeli\\speedrun-lulus\\dataset\\preprocessed\\test_preprocessed_smoke.csv.gz'},
 'artifacts': {'tr

## 8. Sanity check akhir

Cell ini membaca sampel hasil preprocessing sebagai validasi cepat bahwa output benar-benar terbentuk.


In [11]:
train_preview = pd.read_csv(TRAIN_OUTPUT_PATH, compression=COMPRESSION, nrows=5)
test_preview = pd.read_csv(TEST_OUTPUT_PATH, compression=COMPRESSION, nrows=5)

print("train preview shape:", train_preview.shape)
print("test preview shape: ", test_preview.shape)
train_preview.head()


train preview shape: (5, 49)
test preview shape:  (5, 49)


,id,item_id,dept_id,cat_id,store_id,state_id,store_cat,store_dept,date,d,wm_yr_wk,weekday,wday,dayofweek,month,...,item_id_mean_enc,dept_id_mean_enc,cat_id_mean_enc,store_id_mean_enc,state_id_mean_enc,store_cat_mean_enc,store_dept_mean_enc,event_type_1__Cultural,event_type_1__National,event_type_1__No_Type,event_type_1__Religious,event_type_1__Sporting,event_type_2__Cultural,event_type_2__No_Type,event_type_2__Religious
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,CA_1|HOBBIES,CA_1|HOBBIES_1,2011-01-29,d_1,11101,Saturday,1,5,1,...,0.314602,0.681443,0.681443,0.681443,0.681443,0.681443,0.681443,0,0,1,0,0,0,1,0
1,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,CA_1|HOBBIES,CA_1|HOBBIES_1,2011-01-30,d_2,11101,Sunday,2,6,1,...,0.314602,0.681443,0.681443,0.681443,0.681443,0.681443,0.681443,0,0,1,0,0,0,1,0
2,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,CA_1|HOBBIES,CA_1|HOBBIES_1,2011-01-31,d_3,11101,Monday,3,0,1,...,0.314602,0.681443,0.681443,0.681443,0.681443,0.681443,0.681443,0,0,1,0,0,0,1,0
3,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,CA_1|HOBBIES,CA_1|HOBBIES_1,2011-02-01,d_4,11101,Tuesday,4,1,2,...,0.314602,0.681443,0.681443,0.681443,0.681443,0.681443,0.681443,0,0,1,0,0,0,1,0
4,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,CA_1|HOBBIES,CA_1|HOBBIES_1,2011-02-02,d_5,11101,Wednesday,5,2,2,...,0.314602,0.681443,0.681443,0.681443,0.681443,0.681443,0.681443,0,0,1,0,0,0,1,0
